# Урок 01 - Въведение в AI агенти

Добре дошли в първия урок от курса **AI агенти за начинаещи**!

**AI агент** е програма, която използва голям езиков модел (LLM) като двигател на разсъжденията си и може да предприема *действия* в реалния свят — да извиква API-та, да прави заявки към бази данни или да изпълнява код — за да постигне цел от името на потребител.

В тази тетрадка ще изградите първия си агент: **туристически агент**, който препоръчва ваканционни дестинации. По пътя ще научите как да:

1. Свържете се със службата Microsoft Foundry Agent Service чрез **Microsoft Agent Framework**.
2. Дадете на агента **инструмент** — обикновена Python функция, която може да извиква.
3. Стартирате агента и да разгледате отговора му.
4. Предавате отговора на агента потоково, токен по токен.


## Настройка

Преди да стартирате тази тетрадка, уверете се, че имате:

1. **Проект Microsoft Foundry** с внедрен чат модел (например `gpt-4.1-mini`).
2. **Влезли сте с Azure CLI** — изпълнете `az login` в терминала си.
3. **Зададени са необходимите променливи на средата:**
   - `AZURE_AI_PROJECT_ENDPOINT` — крайна точка на вашия проект Microsoft Foundry.
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — името на внедрения ви модел.

Следващата клетка инсталира необходимите Python пакети.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import dotenv
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from agent_framework import tool

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

if not endpoint or not model:
    raise ValueError(
        "Missing required environment variables. "
        "Please set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME in your .env file."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=AzureCliCredential()
)

## Създаване на Вашия Първи Агент

Агентът се нуждае от две неща:

- **Инструкции**, които му казват *кой е* и *как да се държи* (системен подсказване).
- **Инструменти** — Python функции, декорирани с `@tool`, които агентът може да използва, за да получава информация или изпълнява действия.

По-долу дефинираме прост инструмент, който връща списък с популярни ваканционни дестинации. Агентът ще използва този инструмент, когато потребителят поиска препоръки за пътуване.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [ ]:
agent = provider.as_agent(
    name="TravelAgent",
    instructions=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
    tools=[get_destinations],
)

response = await agent.run(
    "I'm looking for a warm beach destination. What do you recommend?"
)
print(response)

## Отговаряне чрез стрийминг

За по-интерактивно изживяване можете да **стриймвате** отговора на агента. Вместо да чакате пълния отговор, агентът предава части от текста, докато те се генерират. Това е особено полезно в чат интерфейси, където искате да показвате резултатите в реално време.


In [ ]:
async for chunk in agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    print(chunk, end="", flush=True)

## Обобщение

В този урок научихте как да:

- **Създадете доставчик**, който се свързва със службата Microsoft Foundry Agent чрез `FoundryChatClient`.
- **Дефинирате инструмент** с помощта на декоратора `@tool`, така че агентът да може да извиква вашите Python функции.
- **Стартирате агента** с потребителско съобщение и да отпечатате неговия отговор.
- **Поточно предавате отговорите** за изход в реално време.

В следващия урок ще изследваме по-задълбочено агентските рамки и ще научим как да предоставим на агентите по-мощни инструменти и възможности за многоетапно разсъждение.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Отказ от отговорност**:
Този документ е преведен с помощта на AI преводачески услуга [Co-op Translator](https://github.com/Azure/co-op-translator). Въпреки че се стремим към точност, моля имайте предвид, че автоматизираните преводи могат да съдържат грешки или неточности. Оригиналният документ на неговия роден език трябва да се счита за авторитетен източник. За критична информация се препоръчва професионален човешки превод. Ние не носим отговорност за каквито и да е недоразумения или неправилни тълкувания, произтичащи от използването на този превод.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
